# RMSNorm in half precision, the way LLMs do it

RMSNorm rescales each vector by its root-mean-square, with no mean subtraction and no bias:

$$y = \frac{x}{\sqrt{\frac{1}{d}\sum_{k=1}^{d} x_k^2 + \varepsilon}} \odot g$$

LLMs run in `float16` or `bfloat16`, but the sum of squares is computed in `float32`: in `float16`, any value above about 256 squares past the largest finite number (65504) and the norm becomes `inf`.

Write `rmsnorm(x, weight, eps=1e-6, offset=False)` in PyTorch.

## Inputs
- `x`: tensor `(..., d)` of dtype `float16`, `float32` or `float64`. Normalise along the last axis.
- `weight`: `(d,)`, same dtype as `x`.
- `offset`: if `True`, the gain is $1 + g$ instead of $g$ (a convention used by some model families, so that a zero-initialised weight means identity).

## Rules
1. Compute in `float32` when `x` is `float16`; otherwise compute in `x`'s own dtype. Cast `weight` to the compute dtype too.
2. Apply the gain in the compute dtype, then cast the result back to `x.dtype`.
3. $\varepsilon$ goes **inside** the square root.

## Output
A tensor of the same shape and dtype as `x`. Use tensor ops only.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def rmsnorm(x, weight, eps=1e-6, offset=False):
    """Normalise x along its last axis by its RMS, times the gain. Same shape and dtype as x."""
    rms = torch.sqrt((x * x).mean(dim=-1, keepdim=True) + eps)   # TODO: overflows in float16, ignores offset
    return x / rms * weight


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[3.0, 4.0], [1.0, -1.0]], "dtype": "float64"}, {"__nd__": [1.0, 2.0], "dtype": "float64"}], "e": {"__nd__": [[0.8485281034827337, 2.2627416092872896], [0.999999500000375, -1.99999900000075]], "dtype": "float64"}}, {"a": [{"__nd__": [[3.0, 4.0]], "dtype": "float64"}, {"__nd__": [0.0, 0.5], "dtype": "float64"}, 1e-06, true], "e": {"__nd__": [[0.8485281034827337, 1.6970562069654673]], "dtype": "float64"}}, {"a": [{"__nd__": [[300.0, -400.0, 250.0, 100.0]], "dtype": "float16"}, {"__nd__": [1.0, 1.0, 1.0, 1.0], "dtype": "float16"}], "e": {"__nd__": [[1.056640625, -1.4091796875, 0.88037109375, 0.352294921875]], "dtype": "float16"}}, {"a": [{"__nd__": [[[1020.5, -1278.0, 209.0, -284.0, -226.375, -107.8125, -1010.0, -115.9375, -432.5, 1661.0, 112.875, -176.375, -140.625, -334.0, -527.5, -195.375], [241.0, -119.25, 479.0, -99.875, 12.1328125, 773.0, 272.5, -252.625, -91.4375, 270.25, 967.5, -134.75, -121.75, 501.25, -443.25, -145.875], [441.25, 290.25, 45.75, 335.0, -1414.0, 510.75, -479.75, -834.5, 138.25, 350.25, -222.375, -538.0, 13.0625, -26.375, 703.0, 373.75]], [[96.9375, 556.0, -102.75, -463.0, 292.0, 291.25, -107.4375, -391.5, 114.5625, -1247.0, 345.0, 245.625, -819.5, 30.671875, -482.0, 378.5], [-1017.0, -457.25, 354.75, 578.0, -1079.0, -249.0, 164.0, -304.5, 795.5, -595.5, 177.25, -524.0, 703.0, -10.828125, -186.125, -859.0], [841.0, 376.5, 376.75, 569.0, 174.625, -319.5, -400.0, -400.0, 685.0, -730.0, -298.25, -160.625, 112.3125, 287.75, -624.5, -865.0]]], "dtype": "float16"}, {"__nd__": [0.99951171875, 1.12109375, 1.076171875, 1.021484375, 0.96826171875, 1.029296875, 0.9755859375, 1.08203125, 0.92041015625, 1.013671875, 0.98876953125, 1.0546875, 1.0224609375, 1.2548828125, 1.1494140625, 1.1494140625], "dtype": "float16"}], "e": {"__nd__": [[[1.5068359375, -2.1171875, 0.332275390625, -0.428466796875, -0.32373046875, -0.1639404296875, -1.455078125, -0.185302734375, -0.587890625, 2.486328125, 0.1649169921875, -0.274658203125, -0.21240234375, -0.619140625, -0.8955078125, -0.331787109375], [0.6015625, -0.333740234375, 1.287109375, -0.254638671875, 0.029327392578125, 1.986328125, 0.66357421875, -0.6826171875, -0.2100830078125, 0.68408203125, 2.388671875, -0.354736328125, -0.310791015625, 1.5703125, -1.2724609375, -0.418701171875], [0.81396484375, 0.6005859375, 0.09088134765625, 0.6318359375, -2.52734375, 0.97021484375, -0.86376953125, -1.6669921875, 0.23486328125, 0.6552734375, -0.40576171875, -1.0478515625, 0.024658203125, -0.06109619140625, 1.4912109375, 0.79296875]], [[0.2021484375, 1.30078125, -0.230712890625, -0.98681640625, 0.58984375, 0.62548828125, -0.21875, -0.8837890625, 0.2200927734375, -2.638671875, 0.7119140625, 0.54052734375, -1.7490234375, 0.080322265625, -1.15625, 0.90771484375], [-1.716796875, -0.86572265625, 0.64453125, 0.9970703125, -1.7646484375, -0.432861328125, 0.270263671875, -0.556640625, 1.236328125, -1.01953125, 0.2958984375, -0.93359375, 1.2138671875, -0.02294921875, -0.361328125, -1.66796875], [1.6572265625, 0.83203125, 0.79931640625, 1.1455078125, 0.333251953125, -0.6484375, -0.76904296875, -0.85302734375, 1.2431640625, -1.458984375, -0.58154296875, -0.333984375, 0.226318359375, 0.7119140625, -1.4150390625, -1.9599609375]]], "dtype": "float16"}}, {"a": [{"__nd__": [[-2.039503812789917, -0.34031662344932556, -0.6086106300354004, 0.5327215790748596, -2.279026508331299, 1.1744986772537231, 1.0669833421707153, -1.3020708560943604], [-0.978548526763916, -0.8011720180511475, 0.04329590126872063, 0.6409710645675659, 2.0478861331939697, -0.19744543731212616, 0.7675025463104248, 0.1554178148508072], [1.7599263191223145, 0.7421578764915466, 1.368550419807434, -1.077675223350525, -0.19224071502685547, -0.813772439956665, 1.5049474239349365, 0.6576399207115173], [-0.30514442920684814, -0.4524678885936737, 0.4846648871898651, -0.7014955282211304, -0.9305889010429382, 0.48127448558807373, 2.46313214302063, -0.24613355100154877]], "dtype": "float32"}, {"__nd__": [-0.5558657646179199, -1.171156883239746, -1.3350110054016113, 0.5249832272529602, 0.850803017616272, 0.00917472131550312, 0.3325759768486023, 0.11591657996177673], "dtype": "float32"}, 1e-05, true], "e": {"__nd__": [[-0.6766370534896851, 0.043510548770427704, 0.1523055136203766, 0.6068513989448547, -3.1508419513702393, 0.8853926658630371, 1.062103033065796, -1.0853838920593262], [-0.46939054131507874, 0.14810095727443695, -0.01566547527909279, 1.0557016134262085, 4.093584060668945, -0.2152044177055359, 1.1046115159988403, 0.1873140037059784], [0.6963754296302795, -0.11316844820976257, -0.4084647595882416, -1.464156985282898, -0.3169861137866974, -0.7316510677337646, 1.7866851091384888, 0.6538145542144775], [-0.13345474004745483, 0.07625994086265564, -0.1598876714706421, -1.0534266233444214, -1.6960256099700928, 0.47827044129371643, 3.232168674468994, -0.2704686224460602]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, {"__nd__": [1.0, 1.0, 1.0, 1.0], "dtype": "float64"}, 1e-06], "e": {"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}}, {"a": [{"__nd__": [[0.0001, 0.0001, 0.0001]], "dtype": "float64"}, {"__nd__": [1.0, 1.0, 1.0], "dtype": "float64"}, 1e-06], "e": {"__nd__": [[0.09950371902099892, 0.09950371902099892, 0.09950371902099892]], "dtype": "float64"}}, {"a": [{"__nd__": [[[0.13865484604722084, -1.5261590525828475, -0.45811827304922115, 0.11147930758994773, -0.7831667805636854, -0.47642974385981746, -0.8191201803277345], [-0.3334966741984413, 0.8531082743251359, -0.40658035065559817, -0.15387068019710476, 0.813718359438118, 0.6447702141569163, 1.6952075589596332], [-2.090485078128588, 0.856858944523106, -0.4822840867730822, 0.13469181881788084, 0.837719384899064, 1.0832531556913447, 1.0393506903088359], [0.15510669107825276, 1.6096626830183138, -0.2829742272161367, -0.14098191307078672, 0.7993511888940862, -0.5513724079804637, 2.160906258701247], [1.019206565354622, 2.1755753226203938, -0.026589187155779802, -0.38308855026130717, 0.16704800216692559, 0.7345765026061907, -0.5874276868740115]], [[0.3797017469724474, -0.01680427892083426, 1.6156817605764437, -0.6627018123008657, 1.046161771728719, -0.6438635355400087, -0.9606341931074224], [-0.7102994273756232, -1.1901858618422472, 0.1463503600664521, 1.0312607960786646, 0.1643148785347983, 0.6243251121848756, 1.6321741955751323], [0.27002644717885166, 0.19516236096309458, -0.27506146796027464, -1.6082422850112825, 0.7597389318680837, -1.7564604476259789, 0.6526764363838281], [-0.014320335733954404, 1.1267814508104348, -0.067770260510462, -0.8232001579576669, 0.3579102667525859, -0.5602361357081261, -0.1809568246618137], [0.0418707868620003, -0.13454971174566932, -0.18880227497381663, -0.8324449882870513, -0.18913378409281006, -2.13834060712102, -0.15733209522523053]], [[-1.198055985931922, 1.1202636903470218, 1.2699267473212534, -1.951025780521256, 0.14491679036588978, -0.12636392216675604, -1.0467156300808687], [0.5316531426489993, -0.46168954465474565, -1.7675990956866041, -0.2666766520569678, -0.14825311830009477, 0.10643101824147437, -1.2312329987327657], [0.6156809269215894, 0.7354570881988916, -1.1458272768549103, -0.6588555689831765, -0.08033731382935666, -0.5659253270123664, 1.7437546098496857], [0.20837211615824647, -1.010582036354199, -0.7881089276247821, -0.05747025825572099, 2.2957788082586084, -0.17826432182306642, 0.12747641923980615], [0.5140453781581755, -0.0401259362630417, 2.280660872566708, -0.5315249470251602, 0.7442031251457859, 0.16043796898357113, 0.6211963732164166]]], "dtype": "float64"}, {"__nd__": [-1.1766429584197382, 1.768112695107191, -0.12152750716346146, 0.03416326206751611, -0.9349762618687502, -0.7296952681832494, 0.5642394331680308], "dtype": "float64"}, 1e-06, false], "e": {"__nd__": [[[-0.21364511408593392, -3.5336452828170763, 0.07290636014373192, 0.0049873150918130355, 0.9588883775092794, 0.4552538310557824, -0.6052353310010816], [0.4652517600589753, 1.788405158897139, 0.0585831598594277, -0.006232567969792046, -0.9020414966946041, -0.5578254364908786, 1.1340654827492584], [2.2574913375919614, 1.3904443044264818, 0.05379127549624613, 0.004223134083753543, -0.7188420438981468, -0.7254472076411534, 0.5382200396102158], [-0.1673271582247831, 2.6093721357180293, 0.03152917987842058, -0.004415846151250595, -0.685219024231893, 0.36887375892894064, 1.1178681024770651], [-1.2139010355222686, 3.893681606804301, 0.0032708152929277153, -0.013247528840126498, -0.1580950382194396, -0.5425689436717552, -0.33550130652206406]], [[-0.49835685299461385, -0.03314232433950008, -0.21901988724753313, -0.02525402639209853, -1.0910698884652454, 0.5240690343506296, -0.6046090099725256], [0.8962762545566887, -2.2567344328403705, -0.019073224064274807, 0.03778188019201761, -0.16475295042011523, -0.48854892255159393, 0.9876104302803724], [-0.32091640621959255, 0.3485354227572858, 0.033763329228144096, -0.05549472796517676, -0.7174735318115708, 1.2945559417582988, 0.3719651789330879], [0.028617327846772865, 3.3836139766956967, 0.013987655366510098, -0.04776347943403782, -0.568336986224319, 0.6942946625913881, -0.17340818005172431], [-0.0561892401678142, -0.27132514935533636, 0.02616851927927382, -0.03243487377149851, 0.20168195064790137, 1.7795727624480226, -0.10124605204767152]], [[1.2268309084540088, 1.7238246942865938, -0.13431241419447343, -0.058007645229106355, -0.11791856536788713, 0.08024676299568508, -0.5139904375285056], [-0.7229899844301034, -0.9434506274929203, 0.24826617648484348, -0.010529397524000107, 0.1602004077638117, -0.08975712694326751, -0.8029026231037], [-0.7802485339831656, 1.4005539468223946, 0.14997756818498226, -0.024242762953960637, 0.08090035969363354, 0.4447676810772283, 1.059696169837306], [-0.24504398457989998, -1.7858347104902987, 0.09572394312077508, -0.0019622856347752643, -2.145311549086727, 0.1300066910396518, 0.07188744257933653], [-0.6175382873820994, -0.07243573172075152, -0.2829782337161257, -0.01853961518445791, -0.7104112075525931, -0.11952710910112248, 0.3578574609454099]]], "dtype": "float64"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(rmsnorm, json.loads(_TESTS), 'torch', *[0.002, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")